# Demo 2 · частина 1: дальність детекції

До слайда **«Як порахувати дальність детекції»**.

Калькулятор дальності: змінюєш параметри — бачиш, як змінюється дальність. Детектор не потрібен, вистачить базового середовища.

Далі — [частина 2](lesson_01_demo2_part2_detection_threshold.ipynb): перевіряємо поріг 16 px справжнім YOLO.

> Формули — у клітинках нижче, віджети — у `panels.py`. Змінив формулу → перезапусти її клітинку **і клітинку з панеллю**: панель бере функції в момент запуску. Змінив `panels.py` → перезапусти ядро.

## Рівняння проєкції

Ціль висотою `H` на дистанції `D` займає на матриці `f · H / D`. Ділимо на крок пікселя `p` — отримуємо пікселі матриці; ділимо ще на `k` (стиснення кадру до входу моделі) — пікселі на вході моделі. Саме їх порівнюємо з порогом детекції `N`:

```
N = 1000 · f · H / (p · k · D)        ⇔        D = 1000 · f · H / (p · k · N)
```

`f` — мм, `H` і `D` — м, `p` — мкм; множник 1000 переводить мкм у мм.

Добуток `N · D` сталий: удвічі далі — удвічі менше пікселів. Тому рівняння працює в обидва боки, і пару «сенсор + об'єктив» підбирають під потрібну дальність **до** закупівлі.

Ціна дальності — поле зору: `FOV = 2 · atan(сторона / 2f)` (та сама формула, що в demo1). Що довша фокусна, то далі бачимо, але вужчий кут.

### Сенсор і режим — не одне й те саме

Сенсор може віддавати кадр у різних **режимах**, і вони по-різному впливають на формулу:

- **біннінг 2×2** — чотири пікселі зливаються в один: на виході `p` удвічі більший, FOV той самий;
- **кроп** — зчитується лише частина матриці: FOV вужчий.

Тому FOV рахуємо з **області зчитування** (`пікселі · p`), а `p` і `k` у формулі дальності — з **виходу** режиму. Приклад — IMX477 (HQ Camera): 4056×3040 і 2028×1520 мають однаковий FOV, а 1332×990 — вужчий.

In [1]:
import numpy as np

UM_PER_MM = 1000  # p задано в мкм, а f — у мм


def pixels_times_distance(target_m: float, focal_mm: float, pixel_um: float, resize_k: float) -> float:
    """N · D — стала для конфігурації: з неї однаково дістаються і пікселі, і дальність."""
    return UM_PER_MM * focal_mm * target_m / (pixel_um * resize_k)


def pixels_on_target(target_m: float, focal_mm: float, pixel_um: float,
                     resize_k: float, distance_m: float) -> float:
    """Скільки пікселів займе ціль на вході моделі."""
    return pixels_times_distance(target_m, focal_mm, pixel_um, resize_k) / distance_m


def detection_range(target_m: float, focal_mm: float, pixel_um: float,
                    resize_k: float, min_px: float) -> float:
    """Дистанція, на якій ціль ще займає min_px пікселів."""
    return pixels_times_distance(target_m, focal_mm, pixel_um, resize_k) / min_px


def focal_for_range(target_m: float, pixel_um: float, resize_k: float,
                    min_px: float, distance_m: float) -> float:
    """Зворотна задача: яка фокусна потрібна, щоб дістати задану дальність."""
    return pixel_um * resize_k * min_px * distance_m / (UM_PER_MM * target_m)


def field_of_view(frame_px: int, pixel_um: float, focal_mm: float) -> float:
    """Кут огляду (°) уздовж frame_px пікселів сенсора з кроком p (для кропу — лише зчитана частина)."""
    sensor_mm = frame_px * pixel_um / UM_PER_MM
    return 2 * np.degrees(np.arctan(sensor_mm / (2 * focal_mm)))


def effective_pixel(pixel_um: float, readout_px: int, output_px: int) -> float:
    """Крок пікселя на виході режиму: біннінг зливає readout_px / output_px пікселів сенсора в один."""
    return pixel_um * readout_px / output_px


def resize_factor(frame_w_px: int, model_input_px: int) -> float:
    """У скільки разів кадр стискається до входу моделі. Tiled inference прибирає це стиснення (k = 1)."""
    return frame_w_px / model_input_px

### Звірка зі слайдом

Три сценарії з презентації: ціль 2.5 м, піксель 2.9 мкм, поріг 16 px.

In [2]:
for name, focal, k in [("Базовий (1080p → 640)", 25, 3.0),
                       ("Tiled / SAHI", 25, 1.0),
                       ("Tiled + теле", 50, 1.0)]:
    print(f"{name:<24} f={focal:>3} мм  k={k:<4} →  {detection_range(2.5, focal, 2.9, k, 16):>6.0f} м")

Базовий (1080p → 640)    f= 25 мм  k=3.0  →     449 м
Tiled / SAHI             f= 25 мм  k=1.0  →    1347 м
Tiled + теле             f= 50 мм  k=1.0  →    2694 м


## Калькулятор

**Зліва** — пікселі на цілі залежно від відстані; перетин із пунктиром порогу — дальність детекції. **Справа** — що дає кожен важіль порівняно з поточною конфігурацією. FOV видно біля повзунка фокусної, під заголовком (разом зі смугою, яку кадр охоплює на D max) і біля кожного важеля.

Спробуй:

1. Увімкни **Tiled inference** — дальність ×3 без зміни обладнання.
2. Подвой **f** — ще ×2, але FOV звузиться майже вдвічі.
3. Порівняй: подвоєння входу моделі теж дає ×2, але коштує приблизно ×4 обчислень на кадр.
4. Обери **IMX477** і перемикай режими. 4056×3040 і 2028×1520 дають однакову дальність: кадр однаково стискається до входу моделі. Різниця з'являється лише з tiled — біннінг удвічі зменшує дальність. Кроп 1332×990 звужує FOV і збільшує дальність.

In [3]:
from panels import launch_range

launch_range(pixels_on_target, detection_range, focal_for_range, resize_factor,
             field_of_view, effective_pixel)